# 面试问题：高并发服务的限流应该怎样设计？Token Bucket、Leaky Bucket 和滑动窗口如何取舍？

**一句话回答**：先区分要保护的是请求数、token/GPU 时间还是下游并发，再定义允许突发、长期速率、层级配额和超限行为。Token Bucket 允许受控 burst，Leaky Bucket 平滑输出，滑动窗口精确限制任意窗口；生产系统还要处理多级原子扣费、单调时钟、分布式一致性、`Retry-After`、优先级与故障时 fail-open/closed。

下面只用标准库实现三种算法、分层限流和可验证 trace，不调用网关或限流组件。

In [ ]:
from dataclasses import dataclass
from collections import Counter, deque
import hashlib, json, math
import numpy as np

SEED85=8501
assert SEED85==8501
assert math.ceil(1.01)==2
assert hashlib.sha256(b"tenant-a").hexdigest()!=hashlib.sha256(b"tenant-b").hexdigest()

## 1. 限流单位与请求合同

对 LLM 服务，1 个短请求和 1 个万 token 请求成本不同，因此 cost 可用预估 input+output token、KV block 或 GPU 毫秒；对普通 API 才常用 cost=1。请求包含 tenant/user/route/时间/优先级/idempotency key。

时间必须单调。系统时钟回拨若被当作负 refill，可能多发 token 或永久封禁；分布式节点应使用服务端时间并限制最大跳变。

In [ ]:
@dataclass(frozen=True)
class Request85:
    request_id:str; tenant:str; user:str; route:str; at:float; cost:float=1.; priority:int=0
    def __post_init__(self):
        if not self.request_id or not self.tenant or not self.user or self.at<0 or not math.isfinite(self.cost) or self.cost<=0 or self.priority not in (0,1): raise ValueError("request_contract")
req85=Request85("r1","t1","u1","/generate",0.,4.)
assert req85.cost==4 and req85.route=="/generate"
try: Request85("","t","u","/",0,1); raise AssertionError("invalid request accepted")
except ValueError as e: assert str(e)=="request_contract"
assert Request85("r2","t","u","/",1).priority==0

## 2. 手写 Token Bucket

bucket 容量 `B` 控制最大 burst，refill rate `r` 控制长期速率。时刻 `t` 的 token 为 `min(B, old + (t-last)*r)`；请求 cost 足够则扣除，否则返回需要等待的时间。拒绝也应推进 refill 时间，但不能扣 token。

允许 cost 大于 1 后，`Retry-After` 是 `(cost-current)/r`；若 cost 永远大于 capacity，应直接拒绝为不可满足请求。

In [ ]:
class TokenBucket85:
    def __init__(self,capacity,rate,now=0.):
        if capacity<=0 or rate<=0: raise ValueError("bucket_contract")
        self.capacity=float(capacity); self.rate=float(rate); self.tokens=float(capacity); self.last=float(now)
    def _project(self,now):
        if now<self.last: raise ValueError("non_monotonic_time")
        return min(self.capacity,self.tokens+(now-self.last)*self.rate)
    def allow(self,now,cost=1.,commit=True):
        if cost<=0 or cost>self.capacity: return False,math.inf
        available=self._project(now); ok=available+1e-12>=cost; retry=0. if ok else (cost-available)/self.rate
        if commit: self.tokens=available-cost if ok else available; self.last=float(now)
        return ok,retry
bucket85=TokenBucket85(5,2)
assert [bucket85.allow(0)[0] for _ in range(5)]==[True]*5 and not bucket85.allow(0)[0]
ok85,retry85=bucket85.allow(.25); assert not ok85 and math.isclose(retry85,.25)
assert bucket85.allow(.5)[0] and math.isclose(bucket85.tokens,0.)

## 3. 精确滑动窗口

保存最近 `W` 秒每次通过的 `(timestamp,cost)`，先移除 `<= now-W` 的事件，再判断窗口总 cost。它严格限制任意窗口，但状态与请求量成正比；生产可用分桶近似或 GCRA 降低内存。

边界必须定义清楚：本例窗口是 `(now-W, now]`，恰好在左边界的事件会过期。

In [ ]:
class SlidingWindow85:
    def __init__(self,window,limit):
        if window<=0 or limit<=0: raise ValueError("window_contract")
        self.window=float(window); self.limit=float(limit); self.events=deque(); self.total=0.; self.last=-math.inf
    def allow(self,now,cost=1.,commit=True):
        if now<self.last: raise ValueError("non_monotonic_time")
        while self.events and self.events[0][0]<=now-self.window:
            _,old=self.events.popleft(); self.total-=old
        ok=cost>0 and self.total+cost<=self.limit
        retry=0. if ok else (self.events[0][0]+self.window-now if self.events else math.inf)
        if commit:
            self.last=now
            if ok: self.events.append((now,cost)); self.total+=cost
        return ok,max(0.,retry)
sw85=SlidingWindow85(10,3)
assert [sw85.allow(0)[0] for _ in range(4)]==[True,True,True,False]
assert not sw85.allow(9.9)[0] and sw85.allow(10)[0]
assert sw85.total==1 and len(sw85.events)==1

## 4. Leaky Bucket 平滑输出

Leaky Bucket 把请求放入有限队列，以固定速率排出，适合保护不能承受 burst 的下游。它改变请求等待时间；Token Bucket 允许立即消耗已有 token。队列满时拒绝，队列未满也可能产生排队延迟。

下例使用“待处理 work”表示水位，随时间按 leak rate 下降。

In [ ]:
class LeakyBucket85:
    def __init__(self,capacity,leak_rate): self.capacity=float(capacity); self.rate=float(leak_rate); self.level=0.; self.last=0.
    def push(self,now,cost=1.):
        if now<self.last or cost<=0: raise ValueError("leaky_contract")
        self.level=max(0.,self.level-(now-self.last)*self.rate); self.last=now
        if self.level+cost>self.capacity: return False,self.level/self.rate
        delay=self.level/self.rate; self.level+=cost; return True,delay
leak85=LeakyBucket85(4,1)
assert leak85.push(0,2)==(True,0.)
ok2_85,delay2_85=leak85.push(0,2); assert ok2_85 and math.isclose(delay2_85,2.)
assert not leak85.push(0,1)[0] and leak85.push(2,1)[0]

## 5. global → tenant → user 分层扣费必须原子

如果先扣 global，再发现 user 超限，会“吃掉”未实际服务的全局额度。正确做法是对所有 bucket preview，全部允许后再以同一时间 commit；分布式实现通常用单 key 脚本、事务或 reservation/rollback。

超限原因应返回最窄层级，便于用户理解；但对外不要泄露其他租户容量。

In [ ]:
class HierarchicalLimiter85:
    def __init__(self,buckets): self.buckets=buckets; self.metrics=Counter()
    def allow(self,keys,now,cost):
        chain=[self.buckets[k] for k in keys]; previews=[b.allow(now,cost,commit=False) for b in chain]
        if not all(x[0] for x in previews):
            idx=next(i for i,x in enumerate(previews) if not x[0]); self.metrics[f"reject:{keys[idx]}"]+=1; return False,max(x[1] for x in previews if not x[0]),keys[idx]
        for b in chain: assert b.allow(now,cost,commit=True)[0]
        self.metrics["allow"]+=1; return True,0.,"ok"
buckets85={"global":TokenBucket85(20,10),"tenant:t1":TokenBucket85(8,4),"user:u1":TokenBucket85(3,1)}; hierarchy85=HierarchicalLimiter85(buckets85)
assert hierarchy85.allow(["global","tenant:t1","user:u1"],0,3)[0]
global_before85=buckets85["global"].tokens; denied85=hierarchy85.allow(["global","tenant:t1","user:u1"],0,1)
assert not denied85[0] and denied85[2]=="user:u1"
assert buckets85["global"].tokens==global_before85

## 6. 优先级要靠保留容量，不靠无限插队

给高优请求一个独立 reserve bucket，普通请求只能使用 shared bucket；高优请求先尝试 shared，失败后再用 reserve。这样应急流量有保障，又不会在平时永久占用全部容量。优先级必须来自可信鉴权信息，不能接受客户端随意填写。

reserve 使用率和普通流量拒绝率要同时监控，避免所谓 VIP 把系统变成两套失控配额。

In [ ]:
class PriorityLimiter85:
    def __init__(self,shared,reserve): self.shared=shared; self.reserve=reserve
    def allow(self,now,cost,priority):
        if self.shared.allow(now,cost)[0]: return True,"shared"
        if priority==1 and self.reserve.allow(now,cost)[0]: return True,"reserve"
        return False,"limited"
priority85=PriorityLimiter85(TokenBucket85(2,.1),TokenBucket85(1,.1))
assert priority85.allow(0,2,0)==(True,"shared")
assert priority85.allow(0,1,0)==(False,"limited")
assert priority85.allow(0,1,1)==(True,"reserve") and priority85.allow(0,1,1)==(False,"limited")

## 7. 用 trace 比较 burst 与长期速率

算法选择要用真实到达 trace 回放：Token Bucket 初始 burst 高、之后按 rate 放行；滑动窗口在边界更严格；Leaky Bucket 通过排队平滑。指标包括 allow/reject、retry-after、排队延迟、per-tenant 成功率与下游过载率。

只看“拒绝率低”会鼓励无限放行，必须同时验证下游容量不被突破。

In [ ]:
arrivals85=[0.]*8+[.5,1,1.5,2,3,4,5]
tb_sim85=TokenBucket85(5,1); sw_sim85=SlidingWindow85(5,5); leak_sim85=LeakyBucket85(5,1)
tb_out85=[tb_sim85.allow(t)[0] for t in arrivals85]; sw_out85=[sw_sim85.allow(t)[0] for t in arrivals85]; leak_out85=[leak_sim85.push(t)[0] for t in arrivals85]
assert sum(tb_out85)>=5 and sum(sw_out85)>=5 and sum(leak_out85)>=5
assert tb_out85[:5]==[True]*5 and not any(tb_out85[5:8])
assert sum(tb_out85)<=5+math.floor(max(arrivals85)*1)+1

## 8. 分布式一致性、故障策略与发布

本地 limiter 延迟低但各实例总额度会放大；中心化 store 更一致却增加网络依赖。可把全局额度按实例租约分片，定期归还；热点 tenant 使用一致 hash 固定到 owner。鉴权/支付通常 fail-closed，低风险搜索可短时 fail-open，但都要设硬上限。

配置是制品：容量、rate、cost estimator、层级、豁免规则和故障策略必须签名、灰度并支持回滚。

In [ ]:
manifest85={"schema":1,"algorithm":"hierarchical_token_bucket","levels":["global","tenant","user"],"cost":"estimated_tokens_v2","fail_mode":"closed_for_generate","clock":"server_monotonic"}
raw85=json.dumps(manifest85,sort_keys=True,separators=(",",":")); digest85=hashlib.sha256(raw85.encode()).hexdigest()
assert len(digest85)==64 and manifest85["levels"][0]=="global"
forged85=dict(manifest85,fail_mode="always_open")
assert hashlib.sha256(json.dumps(forged85,sort_keys=True,separators=(",",":")).encode()).hexdigest()!=digest85
assert hierarchy85.metrics["allow"]==1 and hierarchy85.metrics["reject:user:u1"]==1

## 9. 面试收束、参考与练习

回答闭环：保护目标/成本单位 → burst 与长期速率 → 三种算法 → 多级原子扣费 → Retry-After/公平性 → trace 回放 → 分布式租约 → fail-open/closed 与版本发布。不要只背三个定义。

练习：实现分桶滑动窗口；加入优先级保留容量；模拟 4 个实例各自本地限流造成额度放大；设计 token 预估偏差后的结算/退款。

参考：[RFC 2697 Single Rate Three Color Marker](https://www.rfc-editor.org/rfc/rfc2697)、[RFC 6585 429 状态码](https://www.rfc-editor.org/rfc/rfc6585)、[GCRA 说明 RFC 2698](https://www.rfc-editor.org/rfc/rfc2698)。